# Build NB for MOFA Approach

2026.03.24 
Built by Bryant A. Chambers
Built for Aegis

## Background

Goal: Identify Modules in aeDNA that are driven by multiple levels of prior information. Two approaches are tested in the Network project, PLIER which relies on one level of prior information but collapses the exposure environment into only the prior information and the testing samples and , two, MOFA, which optimizes mutiple levels at once. The two central methods rely on matrix factorization. PLIER uses an older approach: 


$$
Y = CUB + \Delta B + E
$$

Here the assumption is made that there is some matrix, Y, that contains the "expression" of genes in each sample. This matrix can be summarized as loadings B and Z, that is Y = BZ, but the loadings in Z can be summarized as UC so you get Y = CUB with Error. This allows us to take prior information like mappings to KEGG orthologs and identify how much they might contribute to the expression of genes in each of the samples or in our case, the overall metabolism occuring in each of the modules!!! So if we use prior information from KEGG or BGCs we can start to understand the microbiome in this context rather than just correlational networking. This *might* drive module fomation into functional units more easily.

The issue here is that the data is collapsed into the Latent Value representation and would also need some amount of Feature Set Enrichment Analysis (FSEA) on the latent factor loadings.
Instead of doing a standard enrichment on a list of differentially abundant genes, you take the loadings (weights) of every species or KO in a specific Latent Factor and run a GSEA-style test.

This identifies whether a specific Latent Factor is "enriched" for a certain KEGG Pathway or a certain Taxonomic Class. This is the "Species enrichment per latent variable" you're looking for.

A more modern version of this, with bayesian factor analysis, would allow Multiview of mutliple elements of prior knowledge which solves this problem without needing FSEA. MOFA assumes a power of loading from multiple loadings.

$$
Y_m = Z(W_m)^T + \epsilon_m
$$

$Y_m$
 : Your actual data matrix for a specific view m (e.g., your OTU table).

Z: The Latent Factors (The hidden biological drivers operating across all samples). This is a single matrix shared across all views.

$W_m$
 : The Weights (How strongly each specific feature—like a specific KEGG pathway or a specific BGC—is connected to the latent factor). Each view gets its own weight matrix.

$ϵ_m$
 : The residual noise (what the model couldn't explain).

The Goal: MOFA calculates Z and $W_m$
  to explain as much variance in $Y_m$
  as possible.

**Data needed for Study**  
The Data Preparation (Common Formats)  
Before feeding data to the model, it must be formatted and normalized so that one highly abundant view doesn't drown out the others.  
    - View 1: OTUs/ASVs (Taxonomy):  
     Format: Rows = Samples, Columns = Taxa. 
     Prep: Raw counts should be transformed. Centered Log-Ratio (CLR) transformation is the gold standard here to handle the compositional nature of sequencing data.  
    - View 2: KEGG (Functional Potential):  
    Format: Rows = Samples, Columns = KOs or Pathways.  
    Prep: Usually Log2-transformed Transcripts Per Million (TPM) or similar normalized abundances.  
    - View 3: BGCs (Secondary Metabolites):  
    Format: Rows = Samples, Columns = Gene Clusters (e.g., from antiSMASH or BiG-SCAPE).  
    Prep: Binary (presence/absence) or normalized count data. MOFA can handle different statistical distributions (e.g., Poisson for counts, Bernoulli for binary).  
    - Metadata (The Environment):  
    Format: Rows = Samples, Columns = Covariates (Sediment Depth, Site Location, Paleoecosystem classification).  

Note: Metadata is not fed into the initial MOFA training. It is kept aside for the interpretation step.

## Synthetic Data

In [8]:
# Load transformation library
# install.packages("compositions")
library(compositions)

set.seed(42) # For reproducibility


In [12]:

# 1. Setup Experimental Design
# ------------------------------------------------------------
n_samples <- 50
sample_names <- paste0("Sample_", 1:n_samples)

# Create a "Hidden Signal" (e.g., a Sediment Depth gradient)
# This is the "Latent Factor" we hope MOFA will rediscover.
depth_signal <- seq(-2, 2, length.out = n_samples) 

# 2. View 1: Taxonomy (mat_taxa)
# ------------------------------------------------------------
# We'll simulate 100 Genera. 
# Some will increase with depth, some will decrease, others are noise.
n_taxa <- 100
mat_taxa_raw <- matrix(rpois(n_samples * n_taxa, lambda = 10), 
                       nrow = n_taxa, ncol = n_samples)

# Inject signal into the first 20 taxa
for(i in 1:20) {
  mat_taxa_raw[i, ] <- rpois(n_samples, lambda = exp(2 + (depth_signal * runif(1, -1, 1))))
}

rownames(mat_taxa_raw) <- paste0("Genus_", 1:n_taxa)
colnames(mat_taxa_raw) <- sample_names

# CRITICAL STEP: CLR Transformation
# We add a small pseudocount (1) to handle zeros, then CLR.
mat_taxa <- t(clr(t(mat_taxa_raw + 1)))

In [13]:
View(mat_taxa_raw %>% head(10))
View(mat_taxa %>% head(10)) 

,Sample_1,Sample_2,Sample_3,Sample_4,Sample_5,Sample_6,Sample_7,Sample_8,Sample_9,Sample_10,⋯,Sample_41,Sample_42,Sample_43,Sample_44,Sample_45,Sample_46,Sample_47,Sample_48,Sample_49,Sample_50
Genus_1,30,26,28,28,31,21,13,16,15,15,⋯,3,3,3,4,3,1,3,4,2,2
Genus_2,3,2,5,0,3,4,4,0,4,2,⋯,17,17,17,14,15,21,17,23,14,16
Genus_3,5,0,4,2,3,3,4,1,4,2,⋯,15,14,9,23,18,14,16,18,25,25
Genus_4,10,12,13,8,3,6,8,7,12,8,⋯,9,6,1,7,3,4,3,6,4,5
Genus_5,32,27,38,34,29,25,19,21,27,25,⋯,2,4,1,5,3,3,2,2,2,2
Genus_6,39,46,29,30,32,32,27,22,22,34,⋯,1,2,0,0,0,2,2,0,0,0
Genus_7,2,2,2,3,3,2,5,0,1,4,⋯,18,16,14,21,20,29,19,22,20,20
Genus_8,39,50,35,34,27,28,32,22,20,21,⋯,3,2,6,3,0,0,1,3,3,2
Genus_9,3,2,2,4,6,6,8,3,8,3,⋯,13,14,9,18,14,12,13,16,12,15
Genus_10,11,6,5,6,4,9,8,8,4,9,⋯,9,5,3,6,6,9,4,7,5,6


            Sample_1   Sample_2   Sample_3   Sample_4   Sample_5    Sample_6
Genus_1   1.06719045  0.9448896  0.9865778  1.0547500  1.1273578  0.76991548
Genus_2  -0.98050239 -1.2523349 -0.5889586 -2.3125458 -0.9520837 -0.71168906
Genus_3  -0.57503729 -2.3509472 -0.7712802 -1.2139335 -0.9520837 -0.93483262
Genus_4   0.03109852  0.2140021  0.2583393 -0.1153213 -0.9520837 -0.37521683
Genus_5   1.12971081  0.9812573  1.2828436  1.2428022  1.0628193  0.93696956
Genus_6   1.32208270  1.4992004  1.0204793  1.1214414  1.1581295  1.17538058
Genus_7  -1.26818447 -1.2523349 -1.2821058 -0.9262515 -0.9520837 -1.22251469
Genus_8   1.32208270  1.5808784  1.2028009  1.2428022  0.9938264  1.04616885
Genus_9  -0.98050239 -1.2523349 -1.2821058 -0.7031079 -0.3924680 -0.37521683
Genus_10  0.11810989 -0.4050371 -0.5889586 -0.3666357 -0.7289402 -0.01854188
           Sample_7    Sample_8   Sample_9   Sample_10    Sample_11   Sample_12
Genus_1   0.2777646  0.54951936  0.4529863  0.41755262  0.781599999  0.52

In [14]:
#View 2: Metabolism (mat_kegg)
# ------------------------------------------------------------
# 150 KEGG Orthologs (KOs). Let's assume these are continuous (e.g., TPM).
n_kegg <- 150
mat_kegg <- matrix(rnorm(n_samples * n_kegg, mean = 5, sd = 1), 
                   nrow = n_kegg, ncol = n_samples)

# Inject signal: KOs linked to the taxa above
for(i in 1:30) {
  mat_kegg[i, ] <- 5 + (depth_signal * rnorm(1, 2, 0.5)) + rnorm(n_samples, sd = 0.5)
}

rownames(mat_kegg) <- paste0("KO_", sprintf("%05d", 1:n_kegg))
colnames(mat_kegg) <- sample_names

In [15]:
View((mat_kegg %>% head(10)))

,Sample_1,Sample_2,Sample_3,Sample_4,Sample_5,Sample_6,Sample_7,Sample_8,Sample_9,Sample_10,⋯,Sample_41,Sample_42,Sample_43,Sample_44,Sample_45,Sample_46,Sample_47,Sample_48,Sample_49,Sample_50
KO_00001,1.0218859,2.9953929,2.7599247,1.7559375,2.3464931,2.4349864,1.9918414,3.3134401,3.61982922,2.879640,⋯,6.409225,6.585433,6.996869,5.984999,8.233199,7.896428,7.372238,7.922203,7.550135,8.550225
KO_00002,1.7435744,1.4138399,0.3759376,2.1785675,2.0299577,2.5446106,2.2676484,2.0193414,1.91824024,1.863046,⋯,7.406634,6.938539,7.126349,8.533772,8.198023,8.094853,7.982896,7.846835,7.985703,9.562684
KO_00003,-1.3239673,-1.5744343,-0.7611599,-1.1108734,0.2163594,-0.2579077,-0.1707912,0.3854779,-0.08728012,1.062811,⋯,10.049986,9.888130,9.402516,9.738505,10.198230,10.913236,11.815162,11.788778,11.837688,11.977886
KO_00004,2.9648669,3.5819047,3.9458143,3.3571162,3.4361168,3.0742376,3.6550421,3.8366239,3.92236128,3.522164,⋯,6.130576,6.688710,6.680904,6.913110,6.561549,6.305459,6.760619,6.579473,7.809439,7.353050
KO_00005,2.7643206,2.4078104,3.0889393,2.8562109,2.9266046,3.3448322,2.6974703,3.1928438,3.32142665,3.287626,⋯,6.243328,6.641226,6.546906,6.548005,7.060103,7.423165,6.732014,7.358227,7.341012,7.941062
KO_00006,2.9904929,3.6369573,2.5953152,3.4767412,3.3844729,3.4439113,3.7740850,3.7284486,3.13062217,4.095640,⋯,5.204749,6.677651,6.109453,6.541801,6.601747,7.588486,6.625412,6.486140,6.629439,7.937777
KO_00007,-0.3831528,0.8520402,0.5535547,0.3234017,1.2106383,1.2040293,1.9503034,2.3142457,1.69212559,1.428724,⋯,8.264994,8.030158,8.207925,8.254200,7.991596,9.218647,8.119556,10.112483,9.309387,9.877693
KO_00008,1.6383291,0.6811996,0.7115835,1.2269239,1.3729820,2.9471585,2.9935309,2.1010153,2.26669857,2.734630,⋯,7.242270,7.310163,8.387125,8.067279,7.774171,7.731832,8.444387,8.454597,8.534293,9.800768
KO_00009,0.6532374,1.1886472,0.7210666,0.6775347,1.4116269,1.5522625,1.8158705,3.4677813,2.55164299,1.980353,⋯,8.189886,7.832803,8.348238,8.241218,8.787470,9.002078,9.043696,8.889444,9.223598,8.346436
KO_00010,3.1718933,2.8792166,3.4023157,2.5551759,4.5701974,3.1264654,2.9857802,3.5110034,4.26147007,3.120990,⋯,6.481072,6.813281,6.598037,6.354027,6.835703,6.151035,6.619271,6.762415,7.602581,6.252392


In [16]:
# 4. View 3: Biosynthesis (mat_bgc)
# ------------------------------------------------------------
# 50 BGCs. We'll use binary presence/absence (Bernoulli distribution).
n_bgc <- 50
mat_bgc <- matrix(rbinom(n_samples * n_bgc, 1, prob = 0.2), 
                  nrow = n_bgc, ncol = n_samples)

# Inject signal: Certain BGCs appear only at specific depths
for(i in 1:10) {
  prob_signal <- 1 / (1 + exp(-(depth_signal * rnorm(1, 2, 0.5)))) # Logistic mapping
  mat_bgc[i, ] <- rbinom(n_samples, 1, prob = prob_signal)
}

rownames(mat_bgc) <- paste0("BGC_Cluster_", 1:n_bgc)
colnames(mat_bgc) <- sample_names

In [17]:
mat_bgc %>% head(10)

,Sample_1,Sample_2,Sample_3,Sample_4,Sample_5,Sample_6,Sample_7,Sample_8,Sample_9,Sample_10,⋯,Sample_41,Sample_42,Sample_43,Sample_44,Sample_45,Sample_46,Sample_47,Sample_48,Sample_49,Sample_50
BGC_Cluster_1,0,0,0,1,0,0,0,0,0,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_2,0,0,0,0,0,0,0,0,0,1,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_3,0,0,0,0,0,1,0,0,0,0,⋯,1,1,1,0,1,1,1,1,1,1
BGC_Cluster_4,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_5,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_6,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_7,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,1,1,0,1,1,1,1
BGC_Cluster_8,0,0,0,0,0,0,0,0,0,0,⋯,1,1,0,1,1,1,1,1,1,1
BGC_Cluster_9,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,1,1,1,1,1,1,1
BGC_Cluster_10,0,0,0,0,0,0,0,0,0,0,⋯,1,1,1,0,1,1,1,1,1,1


In [18]:
data_list <- list(
  Taxonomy = mat_taxa,
  Metabolism = mat_kegg,
  Biosynthesis = mat_bgc
)

In [19]:
data_list_analysis <- list(
  Taxonomy = mat_taxa %>% unname() %>% as.matrix(),
  Metabolism = mat_kegg %>% unname() %>% as.matrix(),
  Biosynthesis = mat_bgc %>% unname() %>% as.matrix()
)

In [20]:
mat_taxa %>% unname() %>% as.matrix()

1.06719045,0.94488963,0.98657775,1.05475000,1.12735780,0.76991548,0.27776463,0.54951936,0.45298630,0.41755262,⋯,-0.87927771,-0.88514456,-0.90149132,-0.70976999,-0.88916342,-1.56485368,-0.88372875,-0.65605164,-1.208605978,-1.13727733
-0.98050239,-1.25233495,-0.58895861,-2.31254583,-0.95208375,-0.71168906,-0.75185479,-2.28369398,-0.71016451,-1.25642381,⋯,0.62479968,0.61893284,0.60258608,0.38884230,0.49713094,0.83304160,0.62034865,0.91256428,0.400831934,0.59732373
-0.57503729,-2.35094724,-0.77128016,-1.21393354,-0.95208375,-0.93483262,-0.75185479,-1.59054680,-0.71016451,-1.25642381,⋯,0.50701665,0.43661128,0.01479942,0.85884593,0.66898120,0.45004934,0.56319023,0.67894943,0.950878271,1.02220692
0.03109852,0.21400212,0.25833925,-0.11532125,-0.95208375,-0.37521683,-0.16406813,-0.20425244,0.24534693,-0.15781152,⋯,0.03701302,-0.32552877,-1.59463850,-0.23976636,-0.88916342,-0.64856294,-0.88372875,-0.31957940,-0.697780355,-0.44413015
1.12971081,0.98125728,1.28284357,1.24280223,1.06281927,0.93696956,0.63443957,0.80734847,1.01260209,0.90306044,⋯,-1.16695979,-0.66200101,-1.59463850,-0.52744844,-0.88916342,-0.87170650,-1.17141082,-1.16687726,-1.208605978,-1.13727733
1.32208270,1.49920037,1.02047931,1.12144137,1.15812945,1.17538058,0.97091181,0.85180023,0.81589179,1.20031196,⋯,-1.57242489,-1.17282663,-2.28778568,-2.31920790,-2.27545778,-1.15938857,-1.17141082,-2.26548955,-2.307218267,-2.23588962
-1.26818447,-1.25233495,-1.28210579,-0.92625147,-0.95208375,-1.22251469,-0.56953323,-2.28369398,-1.62645524,-0.74559819,⋯,0.67886690,0.56177442,0.42026452,0.77183455,0.76906466,1.14319653,0.72570916,0.87000466,0.737304171,0.80863282
1.32208270,1.58087840,1.20280086,1.24280223,0.99382640,1.04616885,1.13521486,0.85180023,0.72492001,0.73600635,⋯,-0.87927771,-1.17282663,-0.34187553,-0.93291354,-2.27545778,-2.25800086,-1.57687593,-0.87919519,-0.920923906,-1.13727733
-0.98050239,-1.25233495,-1.28210579,-0.70310792,-0.39246796,-0.37521683,-0.16406813,-0.89739962,-0.12237785,-0.96874174,⋯,0.37348526,0.43661128,0.01479942,0.62523107,0.43259242,0.30694850,0.36903422,0.56772379,0.257731090,0.53669911
0.11810989,-0.40503709,-0.58895861,-0.36663568,-0.72894019,-0.01854188,-0.16406813,-0.08646941,-0.71016451,-0.05245101,⋯,0.03701302,-0.47967945,-0.90149132,-0.37329776,-0.32954763,0.04458424,-0.66058520,-0.18604801,-0.515458798,-0.28997947
-0.75735884,-1.65780005,-0.99442372,-0.92625147,-0.95208375,-0.93483262,-1.26268041,-0.49193451,-1.22099014,-1.25642381,⋯,0.99252446,0.98665762,1.11341170,1.14652800,1.22104978,1.43087860,1.34089480,1.39807209,1.276300671,1.34762932


## Real Data Prep
### From Phyloseq Object
This section outlines the steps necessary for integration of `Phyloseq` objects with the `MOFA` pipeline.

Steps necessary and functions to call:

I. OTU table preparation

Note you need to source all functions from `lib/`
1. read in the phyloseq object with `readRDS()`
2. simultaneously convert and bin at genus or species rank using `phyloseq_conversion.R`
3. transform the matrix `mat_taxa_raw = t(mofa_prep$OTU)` so that it is taxa x samples
4. then perform the clr transform on the extracted OTU table using `clr_transform.R`

II. KO mapping 
TESTING USING HUMAN 3

III. BGC CPM of +/-
SEE SAM

IV. Integration of metabolic potential

V. Eukaryotic OTUs - get else where?!

Integrate into a viewing frame

Run:

`Buldscript.R`
Need to add GPU acceleration!!!!
add cudapy and register dual GPUs




## MOFA Analysis



In [21]:
library(MOFA2)
library(dplyr)
library(tidyr)
library(igraph)
library(reticulate)

#reticulate::use_condaenv("amod_mofapy2")
reticulate::use_python("/maps/projects/caeg/people/gfx654/miniforge3/envs/amod_mofapy2/bin/python", required = TRUE)

In [ ]:
lapply(data_list_analysis, View)

In [22]:
MOFAobject <- create_mofa(data_list_analysis)


Creating MOFA object from a list of matrices (features as rows, sample as columns)...


Warning message in create_mofa_from_matrix(data, groups):
“Feature names are not specified for view 1, using default: feature1_v1, feature2_v1...”
Warning message in create_mofa_from_matrix(data, groups):
“Feature names are not specified for view 2, using default: feature1_v2, feature2_v2...”
Warning message in create_mofa_from_matrix(data, groups):
“Feature names are not specified for view 3, using default: feature1_v3, feature2_v3...”
Warning message in create_mofa_from_matrix(data, groups):
“Sample names for group 1 are not specified, using default: sample1_g1, sample2_g1,...”


In [23]:
# Set Data Options: Tell MOFA what kind of statistical distributions to expect.
# We use "gaussian" for continuous/normalized data.
data_opts <- get_default_data_options(MOFAobject)
data_opts$scale_views <- TRUE # Scales views so one doesn't dominate the model


In [24]:

# Set Model Options: How many factors (hidden drivers) should it look for?
model_opts <- get_default_model_options(MOFAobject)
model_opts$num_factors <- 10 # A good starting point. MOFA will drop inactive ones.


In [25]:

# Set Training Options
train_opts <- get_default_training_options(MOFAobject)
train_opts$convergence_mode <- "medium" 

# Prepare and Train (This connects to Python's mofapy2 under the hood)
MOFAobject <- prepare_mofa(MOFAobject, data_opts, model_opts, train_opts)
MOFAobject <- run_mofa(MOFAobject, use_basilisk = FALSE)


Checking data options...

Checking training options...

Checking model options...

Warning message in run_mofa(MOFAobject, use_basilisk = FALSE):
“No output filename provided. Using /tmp/Rtmp5xkVrv/mofa_20260324-145226.hdf5 to store the trained model.

”
Connecting to the mofapy2 python package using reticulate (use_basilisk = FALSE)... 
    Please make sure to manually specify the right python binary when loading R with reticulate::use_python(..., force=TRUE) or the right conda environment with reticulate::use_condaenv(..., force=TRUE)
    If you prefer to let us automatically install a conda environment with 'mofapy2' installed using the 'basilisk' package, please use the argument 'use_basilisk = TRUE'


5 factors were found to explain no variance and they were removed for downstream analysis. You can disable this option by setting load_model(..., remove_inactive_factors = FALSE)

Warning message in .quality_control(object, verbose = verbose):
“Factor(s) 1 are strongly correlated wit

In [26]:

# 4. Interpret and Select a Factor
# ------------------------------------------------------------------------------
# In a real workflow, you would correlate the factors with your metadata here.
# e.g., Correlate Factor scores with "Sediment Depth" or "Paleoecosystem State".
# Let's assume we found that Factor 3 strongly correlates with a key environmental shift.
target_factor <- "Factor1"


In [27]:

# 5. Extract Weights to Build the Network (Node & Edge Lists)
# ------------------------------------------------------------------------------
# We extract the weights (loadings) for all features across all views for Factor 3.
weights <- get_weights(MOFAobject, factors = target_factor, as.data.frame = TRUE)

# Filter for the most important features (the "signal").
# We take the absolute value of the weight (magnitude matters, sign dictates direction).
# Here, we keep features in the top 5% of weights for this specific factor.
threshold <- quantile(abs(weights$value), 0.95)
top_features <- weights %>% filter(abs(value) >= threshold)

# Create Node List
# We need to know what view each feature came from so Cytoscape can color them differently.
nodes <- top_features %>%
  select(id = feature, type = view, weight = value) %>%
  distinct()

# Create Edge List
# How do we connect them? In MOFA, if two features are highly weighted on the SAME factor,
# they are co-varying. We create edges between all top features in this factor.
# Note: This creates a fully connected sub-graph (clique) for this factor. 
# For multi-factor networks, you would calculate pairwise correlations of their weight profiles.
edges <- expand.grid(source = nodes$id, target = nodes$id) %>%
  filter(source != target) %>% # Remove self-loops
  # Optional: Remove duplicate undirected edges (A-B and B-A)
  mutate(combo = paste(pmin(source, target), pmax(source, target), sep="_")) %>%
  distinct(combo, .keep_all = TRUE) %>%
  select(source, target)

# Add an edge attribute (e.g., they belong to the Factor 3 interaction module)
edges$module <- target_factor

# 6. Build and Export for Cytoscape
# ------------------------------------------------------------------------------
# Create the igraph object
network_graph <- graph_from_data_frame(d = edges, vertices = nodes, directed = FALSE)

# Export to GraphML (Cytoscape's preferred format)
# In Cytoscape, simply go to File -> Import -> Network from File -> Select this GraphML.
write_graph(network_graph, "MOFA_Factor3_Network.graphml", format = "graphml")

# Alternatively, export clean CSVs for custom knowledge graph ingestion
write.csv(nodes, "MOFA_Node_List.csv", row.names = FALSE)
write.csv(edges, "MOFA_Edge_List.csv", row.names = FALSE)

print("Network construction complete. Ready for Cytoscape or Knowledge Graph integration.")

Warning message:
“There were 2 warnings in `mutate()`.
The first warning was:
ℹ In argument: `combo = paste(pmin(source, target), pmax(source, target), sep =
  "_")`.
Caused by warning in `Ops.factor()`:
! ‘>’ not meaningful for factors
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”


[1] "Network construction complete. Ready for Cytoscape or Knowledge Graph integration."
